# 1. Haar Cascade Classifier (OpenCV)

In [6]:
import cv2
import numpy as np

def haar_face_detection(image_path):
    # Load the cascade classifier
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    
    # Read image
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30),
        flags=cv2.CASCADE_SCALE_IMAGE
    )
    
    # Draw rectangles around faces
    for (x, y, w, h) in faces:
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)
    
    return img, len(faces)

# Usage
result_img, face_count = haar_face_detection('../dataset/test/ronaldo/ronaldo.jpg')
print(f"Detected {face_count} faces")
cv2.imshow('Haar Cascade Detection', result_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

Detected 1 faces


# 2. HOG + SVM (Dlib)

In [9]:
import dlib
import cv2

def hog_face_detection(image_path):
    # Initialize HOG face detector
    detector = dlib.get_frontal_face_detector()
    
    # Load image
    img = cv2.imread(image_path)
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Detect faces
    faces = detector(rgb_img, 1)
    
    # Draw rectangles
    for face in faces:
        x, y, w, h = face.left(), face.top(), face.width(), face.height()
        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
    
    return img, len(faces)

# Usage
result_img, face_count = hog_face_detection('../dataset/test/ronaldo/ronaldo.jpg')
print(f"Detected {face_count} faces using HOG+SVM")
cv2.imshow('HOG + SVM(Dlib)', result_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

Detected 1 faces using HOG+SVM


# 3. CNN-based Face Detector (Dlib)

In [11]:
import dlib
import cv2

def cnn_face_detection(image_path):
    # Load CNN face detector
    cnn_face_detector = dlib.cnn_face_detection_model_v1('mmod_human_face_detector.dat')
    
    img = cv2.imread(image_path)
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Detect faces using CNN
    faces = cnn_face_detector(rgb_img, 1)
    
    for face in faces:
        x, y, w, h = face.rect.left(), face.rect.top(), face.rect.width(), face.rect.height()
        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 0, 255), 2)
        # Confidence score
        cv2.putText(img, f'Conf: {face.confidence:.2f}', (x, y-10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    
    return img, len(faces)

# Usage
result_img, face_count = cnn_face_detection('../dataset/test/ronaldo/ronaldo.jpg')
print(f"Detected {face_count} faces using CNN")
cv2.imshow('CNN-based Face Detector (Dlib)', result_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

Detected 1 faces using CNN


# 4. MTCNN (Multi-task Cascaded CNN)

In [13]:
from mtcnn import MTCNN
import cv2

def mtcnn_face_detection(image_path):
    # Initialize MTCNN detector
    detector = MTCNN()
    
    img = cv2.imread(image_path)
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Detect faces
    results = detector.detect_faces(rgb_img)
    
    for result in results:
        x, y, w, h = result['box']
        confidence = result['confidence']
        
        # Draw bounding box
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 255, 0), 2)
        
        # Draw facial landmarks
        keypoints = result['keypoints']
        for keypoint, coord in keypoints.items():
            cv2.circle(img, coord, 2, (0, 255, 255), -1)
        
        # Confidence text
        cv2.putText(img, f'Conf: {confidence:.2f}', (x, y-10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
    
    return img, len(results)

# Usage
result_img, face_count = mtcnn_face_detection('../dataset/test/ronaldo/ronaldo.jpg')
print(f"Detected {face_count} faces using MTCNN")
cv2.imshow('MTCNN (Multi-task Cascaded CNN)', result_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

Detected 1 faces using MTCNN


# 5. SSD (Single Shot Detector)

In [16]:
import cv2
import numpy as np

def ssd_face_detection(image_path, prototxt='deploy.prototxt', 
                      model='res10_300x300_ssd_iter_140000.caffemodel'):
    # Load SSD model
    net = cv2.dnn.readNetFromCaffe(prototxt, model)
    
    img = cv2.imread(image_path)
    (h, w) = img.shape[:2]
    
    # Preprocess image
    blob = cv2.dnn.blobFromImage(cv2.resize(img, (300, 300)), 1.0, 
                                (300, 300), (104.0, 177.0, 123.0))
    net.setInput(blob)
    detections = net.forward()
    
    face_count = 0
    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        
        if confidence > 0.5:
            face_count += 1
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (x1, y1, x2, y2) = box.astype("int")
            
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), 2)
            cv2.putText(img, f'SSD: {confidence:.2f}', (x1, y1-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    
    return img, face_count

# Usage
result_img, face_count = ssd_face_detection('../dataset/test/ronaldo/ronaldo.jpg')
print(f"Detected {face_count} faces using SSD")
cv2.imshow('SSD (Single Shot Detector)', result_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

Detected 1 faces using SSD


# 6. RetinaFace (InsightFace)

In [22]:
import cv2
import numpy as np
from insightface.app import FaceAnalysis

def retinaface_detection(image_path):
    # Initialize RetinaFace detector
    app = FaceAnalysis(providers=['CPUExecutionProvider'])
    app.prepare(ctx_id=0, det_size=(640, 640))
    
    img = cv2.imread(image_path)
    
    # Detect faces
    faces = app.get(img)
    
    for face in faces:
        bbox = face.bbox.astype(int)
        confidence = face.det_score
        
        # Draw bounding box
        cv2.rectangle(img, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (255, 0, 255), 2)
        
        # Draw landmarks
        landmarks = face.kps.astype(int)
        for i in range(5):
            cv2.circle(img, (landmarks[i][0], landmarks[i][1]), 2, (0, 255, 255), -1)
        
        # Confidence text
        cv2.putText(img, f'Retina: {confidence:.2f}', (bbox[0], bbox[1]-10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 1)
    
    return img, len(faces)

# Usage
result_img, face_count = retinaface_detection('../dataset/test/ronaldo/ronaldo.jpg')
print(f"Detected {face_count} faces using RetinaFace")
cv2.imshow('RetinaFace (InsightFace)', result_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

ModuleNotFoundError: No module named 'insightface'

# 7. YOLO Face Detection

In [26]:
from ultralytics import YOLO
import cv2

def yolo_face_detection(image_path, model_path='yolov8n-face.pt'):
    # Load YOLO model
    model = YOLO(model_path)
    
    # Perform detection
    results = model(image_path)
    
    img = cv2.imread(image_path)
    face_count = 0
    
    for result in results:
        boxes = result.boxes
        for box in boxes:
            face_count += 1
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = box.conf[0]
            
            # Draw bounding box
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 165, 255), 2)
            cv2.putText(img, f'YOLO: {confidence:.2f}', (x1, y1-10), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 1)
    
    return img, face_count

# Usage for YOLOv8
result_img, face_count = yolo_face_detection('../dataset/test/ronaldo/ronaldo.jpg')
print(f"Detected {face_count} faces using YOLO")
cv2.imshow('YOLO Face Detection', result_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

# # For YOLOv11 (similar usage)
# def yolov11_face_detection(image_path):
#     # YOLOv11 follows similar pattern
#     model = YOLO('yolov11n-face.pt')  # Assuming model is available
#     results = model(image_path)
#     # Process results similarly..

ModuleNotFoundError: No module named 'ultralytics'

# 8. MediaPipe Face Detection

In [ ]:
import mediapipe as mp
import cv2

def mediapipe_face_detection(image_path):
    # Initialize MediaPipe Face Detection
    mp_face_detection = mp.solutions.face_detection
    mp_drawing = mp.solutions.drawing_utils
    
    with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:
        img = cv2.imread(image_path)
        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Process image
        results = face_detection.process(rgb_img)
        
        face_count = 0
        if results.detections:
            for detection in results.detections:
                face_count += 1
                
                # Get bounding box
                bboxC = detection.location_data.relative_bounding_box
                h, w, _ = img.shape
                x = int(bboxC.xmin * w)
                y = int(bboxC.ymin * h)
                width = int(bboxC.width * w)
                height = int(bboxC.height * h)
                
                # Draw bounding box
                cv2.rectangle(img, (x, y), (x+width, y+height), (128, 0, 128), 2)
                
                # Draw keypoints
                for keypoint in detection.location_data.relative_keypoints:
                    kx = int(keypoint.x * w)
                    ky = int(keypoint.y * h)
                    cv2.circle(img, (kx, ky), 2, (0, 255, 0), -1)
                
                # Confidence
                confidence = detection.score[0]
                cv2.putText(img, f'MediaPipe: {confidence:.2f}', (x, y-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (128, 0, 128), 1)
        
        return img, face_count

# Usage
result_img, face_count = mediapipe_face_detection('../dataset/test/ronaldo/ronaldo.jpg')
print(f"Detected {face_count} faces using MediaPipe")
cv2.imshow('MediaPipe Face Detection', result_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

# 8. MediaPipe Face Detection